# Tokenization — Luffy GPT Dataset

This notebook covers:
1. How we built and preprocessed our dataset
2. Character-level tokenization
3. Tiktoken (OpenAI BPE)
4. BPE from scratch

Dataset: One Piece dialogue scraped from 271 episodes (One Pace subtitles) + Episode 1 character-tagged transcripts

## 1. Load and Explore the Raw Dataset

In [1]:
with open("luffy_dataset.txt", "r") as f:
    raw = f.read()

print(f"Total characters : {len(raw):,}")
print(f"Total lines      : {raw.count(chr(10)):,}")

Total characters : 3,300,842
Total lines      : 141,089


In [2]:
# Dataset is in TinyShakespeare format: Speaker:\nDialogue\n\n
print(raw[:600])

Gold Roger:
"My treasure? If you want it, you can have it! Find it! I left everything this world has to offer there!"

Marine 1:
"Whoa..."

Captain:
“No worries! A whirlpool of that size poses no problem to this ship!"

Man:
“Miss. May I have this dance?"

Marine 1:
"Alright! Not again..."

Marine 2:
"Batter out!"

Marine 3:
"What're you doing? You suck!"

Marine 1 + 2:
"This thing's heavy! Must be filled with booze then! It's all ours now!"

Marine 3:
"Ship off the starboard stern! There's a pirate flag on its mast! It's a pirate ship! Enemy raid! Enemy raid!"

Marine 1 + 2:
"Captain! Pirates


In [3]:
# Count how many dialogue blocks we have (blocks are separated by double newline)
blocks_raw = [b.strip() for b in raw.split("\n\n") if b.strip()]
print(f"Total dialogue blocks: {len(blocks_raw):,}")
print()
print("Sample block:")
print(blocks_raw[5])

Total dialogue blocks: 5,548

Sample block:
Marine 2:
"Batter out!"


In [4]:
# Who are the speakers? Parse out all unique speaker labels
speakers = []
for block in blocks_raw:
    lines = block.splitlines()
    if lines and lines[0].endswith(":"):
        speakers.append(lines[0][:-1])

from collections import Counter
speaker_counts = Counter(speakers)
print("Top 20 speakers by block count:")
for name, count in speaker_counts.most_common(20):
    print(f"  {name:<20} {count:>5}")

Top 20 speakers by block count:
  Main                  2710
  Thoughts              1067
  Flashback              680
  Secondary              671
  Koby                    51
  Luffy                   50
  Alvida                  24
  Peppoko + Pirate 2       6
  Poppoko                  5
  Peppoko                  5
  Heppoko                  3
  Pirate 2                 3
  Marine 1                 2
  Captain                  2
  Marine 3                 2
  Marine 1 + 2             2
  Pirate 4                 2
  Pirate 3 + 4 + 5         2
  Gold Roger               1
  Man                      1


## 2. Preprocessing

Raw subtitles had issues we had to clean:
- ASS subtitle formatting tags: `{\pos(640,500)\fad(200,200)}`
- Hard line breaks encoded as `\N`
- Unicode smart quotes and em-dashes
- Bracketed stage directions like `[laughing]`
- Lines without any alphabetic content

Below is the `clean_line()` function from our `preprocess_and_split.py`.

In [5]:
import re

URL_RE = re.compile(r"https?://\S+|www\.\S+")
HTML_RE = re.compile(r"<[^>]+>")
WS_RE  = re.compile(r"\s+")

QUOTE_TRANSLATION = str.maketrans({
    "\u201c": '"',   # left double quote
    "\u201d": '"',   # right double quote
    "\u2018": "'",   # left single quote
    "\u2019": "'",   # right single quote
    "\u2014": "-",   # em dash
    "\u2013": "-",   # en dash
    "\u2026": "...", # ellipsis
    "\u00a0": " ",   # non-breaking space
})

def clean_line(line: str) -> str:
    line = line.translate(QUOTE_TRANSLATION)
    line = URL_RE.sub("", line)
    line = HTML_RE.sub("", line)
    line = WS_RE.sub(" ", line).strip()

    # Strip surrounding quotes if the full line is quoted
    if len(line) >= 2 and (
        (line.startswith('"') and line.endswith('"'))
        or (line.startswith("'") and line.endswith("'"))
    ):
        line = line[1:-1].strip()

    # Remove stage directions in brackets (e.g. [laughing])
    line = re.sub(r"^\[[^\]]*\]\s*", "", line)
    line = re.sub(r"\s*\[[^\]]*\]$", "", line).strip()

    # Reject lines with no alphabetic content
    if not re.search(r"[A-Za-z]", line):
        return ""
    return line

In [6]:
# See the cleaning in action on dirty examples from our actual source data
dirty_examples = [
    '\u201cI\'m gonna be King of the Pirates!\u201d',
    '[laughing] You can\'t be serious...',
    'Check https://onepiece.fandom.com for more info',
    'Hm\u2014what do you think?',
    '   too    many    spaces   ',
    '!!!',  # no alphabetic content
]

for ex in dirty_examples:
    cleaned = clean_line(ex)
    print(f"RAW    : {repr(ex)}")
    print(f"CLEANED: {repr(cleaned)}")
    print()

RAW    : "“I'm gonna be King of the Pirates!”"
CLEANED: "I'm gonna be King of the Pirates!"

RAW    : "[laughing] You can't be serious..."
CLEANED: "You can't be serious..."

RAW    : 'Check https://onepiece.fandom.com for more info'
CLEANED: 'Check for more info'

RAW    : 'Hm—what do you think?'
CLEANED: 'Hm-what do you think?'

RAW    : '   too    many    spaces   '
CLEANED: 'too many spaces'

RAW    : '!!!'
CLEANED: ''



In [7]:
# After full preprocessing, load the deduplicated clean corpus
with open("dataset/processed/corpus_clean.txt", "r") as f:
    text = f.read()

blocks_clean = [b.strip() for b in text.split("\n\n") if b.strip()]
print(f"Raw blocks    : {len(blocks_raw):,}")
print(f"Cleaned+deduped: {len(blocks_clean):,}")
print(f"Removed        : {len(blocks_raw) - len(blocks_clean):,} duplicates/empties")
print(f"Corpus size    : {len(text):,} chars")

Raw blocks    : 5,548
Cleaned+deduped: 5,171
Removed        : 377 duplicates/empties
Corpus size    : 3,223,998 chars


## 3. Character-level Tokenization

In [8]:
# All unique characters in the corpus
characters = sorted(list(set(text)))
vocab_size  = len(characters)

print("Vocab (repr so whitespace is visible):")
print(repr("".join(characters)))
print()
print(f"Vocab size: {vocab_size}")
print()
print("Note: TinyShakespeare has 65 chars; we have", vocab_size,
      "— more punctuation from subtitle formatting and Japanese names romanized.")

Vocab (repr so whitespace is visible):
'\n !"#%&\'()+,-.0123456789:;=?ABCDEFGHIJKLMNOPQRSTUVWXYZ\\abcdefghijklmnopqrstuvwxyzÉßàâäèéêîôöûüŒ♪'

Vocab size: 96

Note: TinyShakespeare has 65 chars; we have 96 — more punctuation from subtitle formatting and Japanese names romanized.


In [9]:
# Build encoder / decoder
char_to_idx = {ch: i for i, ch in enumerate(characters)}
idx_to_char = {i: ch for i, ch in enumerate(characters)}

encode = lambda xs: [char_to_idx[x] for x in xs]
decode = lambda xs: ''.join([idx_to_char[x] for x in xs])

In [10]:
print(encode("hello world"))
print(decode(encode("hello world")))

[62, 59, 66, 66, 69, 1, 77, 69, 72, 66, 58]
hello world


In [11]:
# What does a dialogue block look like as tokens?
luffy_line = "Luffy:\nI'm gonna be King of the Pirates!"
encoded = encode(luffy_line)
print("Text  :", repr(luffy_line))
print("Tokens:", encoded)
print("Decoded:", decode(encoded))

Text  : "Luffy:\nI'm gonna be King of the Pirates!"
Tokens: [39, 75, 60, 60, 79, 24, 0, 36, 7, 67, 1, 61, 69, 68, 68, 55, 1, 56, 59, 1, 38, 63, 68, 61, 1, 69, 60, 1, 74, 62, 59, 1, 43, 63, 72, 55, 74, 59, 73, 2]
Decoded: Luffy:
I'm gonna be King of the Pirates!


In [12]:
# Characters that exist in our corpus but NOT in TinyShakespeare
shakespeare_chars = set("\n !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz")
our_extra = set(characters) - shakespeare_chars
print("Extra chars we have (not in Shakespeare):")
print(sorted(our_extra))

Extra chars we have (not in Shakespeare):
['"', '#', '%', '(', ')', '+', '0', '1', '2', '4', '5', '6', '7', '8', '9', '=', '\\', 'É', 'ß', 'à', 'â', 'ä', 'è', 'é', 'ê', 'î', 'ô', 'ö', 'û', 'ü', 'Œ', '♪']


## 4. Tiktoken — OpenAI's BPE Tokenizer

In [13]:
import tiktoken
enc_gpt2 = tiktoken.get_encoding('gpt2')
print("gpt2 vocab size:", enc_gpt2.n_vocab)

gpt2 vocab size: 50257


In [14]:
enc_gpt2.encode("hello world")

[31373, 995]

In [15]:
enc_gpt2.decode([31373, 995])

'hello world'

In [16]:
# Compare char-level vs tiktoken on actual One Piece lines
test_lines = [
    "I'm gonna be King of the Pirates!",
    "Gomu Gomu no Pistol!",
    "I don't care about that. I just want to be free.",
]

for line in test_lines:
    char_toks = encode(line)
    bpe_toks  = enc_gpt2.encode(line)
    print(f"Line   : {line}")
    print(f"Char   : {len(char_toks)} tokens")
    print(f"GPT2   : {len(bpe_toks)} tokens  {bpe_toks}")
    print()

Line   : I'm gonna be King of the Pirates!
Char   : 33 tokens
GPT2   : 9 tokens  [40, 1101, 8066, 307, 2677, 286, 262, 20169, 0]

Line   : Gomu Gomu no Pistol!
Char   : 20 tokens
GPT2   : 9 tokens  [38, 296, 84, 402, 296, 84, 645, 36414, 0]

Line   : I don't care about that. I just want to be free.
Char   : 48 tokens
GPT2   : 14 tokens  [40, 836, 470, 1337, 546, 326, 13, 314, 655, 765, 284, 307, 1479, 13]



In [17]:
# Full corpus comparison (from our tokenizer_sanity_check.py results)
print("=== Tokenization Comparison on train.txt (2.55 MB) ===")
results = [
    ("Character-level", 2_550_667, 96),
    ("gpt2 BPE",        779_136,  50_257),
    ("cl100k_base",     691_908,  100_256),
    ("o200k_base",      655_915,  200_019),
]
for name, tok_count, vocab in results:
    ratio = 2_550_667 / tok_count
    print(f"  {name:<18}  tokens={tok_count:>10,}  chars/token={ratio:.2f}  vocab={vocab:,}")

=== Tokenization Comparison on train.txt (2.55 MB) ===
  Character-level     tokens= 2,550,667  chars/token=1.00  vocab=96
  gpt2 BPE            tokens=   779,136  chars/token=3.27  vocab=50,257
  cl100k_base         tokens=   691,908  chars/token=3.69  vocab=100,256
  o200k_base          tokens=   655,915  chars/token=3.89  vocab=200,019


## 5. BPE from Scratch

Byte-pair encoding iteratively merges the most frequent adjacent token pair into a new single token. We implement it here on a slice of the corpus to keep runtime fast.

In [18]:
def get_stats(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

def merge(ids, pair, idx):
    newids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            newids.append(idx)
            i += 2
        else:
            newids.append(ids[i])
            i += 1
    return newids

In [19]:
# Run BPE on first 200k chars (full 3MB would be slow in the notebook)
text_sample = text[:200_000]

bpe_vocab_size = 40          # target vocab size after merges
num_merges     = vocab_size - bpe_vocab_size
tokens = encode(text_sample)
ids    = list(tokens)

# Build vocab BEFORE the loop so we can show merged strings as we go
vocab  = idx_to_char.copy()  # will grow as we merge
merges = {}  # (int, int) -> int

for i in range(num_merges):
    stats = get_stats(ids)
    pair  = max(stats, key=stats.get)
    idx   = vocab_size + i
    merged_str = vocab[pair[0]] + vocab[pair[1]]  # human-readable using current vocab
    vocab[idx] = merged_str                        # add new token to vocab immediately
    print(f"merging {pair} -> {idx}  ({repr(merged_str)})")
    ids = merge(ids, pair, idx)
    merges[pair] = idx

merging (59, 1) -> 96  ('e ')


merging (2, 0) -> 97  ('!\n')


merging (74, 1) -> 98  ('t ')
merging (74, 62) -> 99  ('th')


merging (69, 75) -> 100  ('ou')
merging (73, 1) -> 101  ('s ')


merging (63, 68) -> 102  ('in')


...

merging (55, 102) -> 143  ('ain')
merging (69, 60) -> 144  ('of')


merging (52, 100) -> 145  ('You')
merging (69, 67) -> 146  ('om')


merging (36, 7) -> 147  ("I'")
merging (99, 96) -> 148  ('the ')
merging (24, 0) -> 149  (':\n')


merging (7, 98) -> 150  ("'t ")


merging (76, 96) -> 151  ('ve ')


In [20]:
print("tokens length:", len(tokens))
print("ids length   :", len(ids))
print(f"compression  : {len(tokens) / len(ids):.2f}X")

tokens length: 200000
ids length   : 133767
compression  : 1.50X


In [21]:
# vocab was already built incrementally in the loop above — just display it
vocab

{0: '\n',
 1: ' ',
 2: '!',
 3: '"',
 4: '#',
 5: '%',
 6: '&',
 7: "'",
 8: '(',
 9: ')',
 10: '+',
 11: ',',
 12: '-',
 13: '.',
 14: '0',
 15: '1',
 16: '2',
 17: '3',
 18: '4',
 19: '5',
 20: '6',
 21: '7',
 22: '8',
 23: '9',
 24: ':',
 25: ';',
 26: '=',
 27: '?',
 28: 'A',
 29: 'B',
 30: 'C',
 31: 'D',
 32: 'E',
 33: 'F',
 34: 'G',
 35: 'H',
 36: 'I',
 37: 'J',
 38: 'K',
 39: 'L',
 40: 'M',
 41: 'N',
 42: 'O',
 43: 'P',
 44: 'Q',
 45: 'R',
 46: 'S',
 47: 'T',
 48: 'U',
 49: 'V',
 50: 'W',
 51: 'X',
 52: 'Y',
 53: 'Z',
 54: '\\',
 55: 'a',
 56: 'b',
 57: 'c',
 58: 'd',
 59: 'e',
 60: 'f',
 61: 'g',
 62: 'h',
 63: 'i',
 64: 'j',
 65: 'k',
 66: 'l',
 67: 'm',
 68: 'n',
 69: 'o',
 70: 'p',
 71: 'q',
 72: 'r',
 73: 's',
 74: 't',
 75: 'u',
 76: 'v',
 77: 'w',
 78: 'x',
 79: 'y',
 80: 'z',
 81: 'É',
 82: 'ß',
 83: 'à',
 84: 'â',
 85: 'ä',
 86: 'è',
 87: 'é',
 88: 'ê',
 89: 'î',
 90: 'ô',
 91: 'ö',
 92: 'û',
 93: 'ü',
 94: 'Œ',
 95: '♪',
 96: 'e ',
 97: '!\n',
 98: 't ',
 99: 'th',
 10

In [22]:
def decode_bpe(ids):
    return "".join(vocab[idx] for idx in ids)

def encode_bpe(text):
    tokens = list(encode(text))
    while len(tokens) >= 2:
        stats = get_stats(tokens)
        pair  = min(stats, key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break
        tokens = merge(tokens, pair, merges[pair])
    return tokens

In [23]:
encode_bpe("hi there")

[62, 63, 118, 105, 59]

In [24]:
encode("hi there")

[62, 63, 1, 74, 62, 59, 72, 59]

In [25]:
# Round-trip check on a real Luffy line
test = "I'm gonna be King of the Pirates!"
decode_bpe(encode_bpe(test)) == test

True

In [26]:
# What are the top BPE merges? They reveal the most common character patterns in One Piece dialogue.
print("Top merges (most frequent bigrams merged first):")
for (p0, p1), idx in merges.items():
    merged_str = vocab[idx]
    print(f"  token {idx}: {repr(merged_str)}")

Top merges (most frequent bigrams merged first):
  token 96: 'e '
  token 97: '!\n'
  token 98: 't '
  token 99: 'th'
  token 100: 'ou'
  token 101: 's '
  token 102: 'in'
  token 103: '..'
  token 104: '.\n'
  token 105: 'er'
  token 106: 'o '
  token 107: 'd '
  token 108: 'an'
  token 109: 'you'
  token 110: 'on'
  token 111: 'at'
  token 112: 'y '
  token 113: ', '
  token 114: 'll'
  token 115: 'ea'
  token 116: 'or'
  token 117: 'ing'
  token 118: ' th'
  token 119: 'a '
  token 120: "'s "
  token 121: 'ar'
  token 122: 'en'
  token 123: '! '
  token 124: 'Th'
  token 125: 'I '
  token 126: 'you '
  token 127: 'is '
  token 128: '...\n'
  token 129: 'at '
  token 130: 'to '
  token 131: '?\n'
  token 132: 'es'
  token 133: 'ow'
  token 134: 'Wh'
  token 135: 'e!\n'
  token 136: 'll '
  token 137: '?!\n'
  token 138: 'ing '
  token 139: 'ir'
  token 140: 'it'
  token 141: 'st'
  token 142: 'gh'
  token 143: 'ain'
  token 144: 'of'
  token 145: 'You'
  token 146: 'om'
  token 147: 